# LoRA Fine-Tuning — Khmer Whisper ASR

**Notebook purpose:** Fine-tune `seanghay/whisper-small-khmer-v2` further using LoRA
(Low-Rank Adaptation) on the expanded training set built in `DATA-09`
(68,059 files: 56,716 raw DDD-Cambodia + 11,343 augmented "phone-mic quality"
copies, on top of the model's already-inherited FLEURS + OpenSLR SLR42 +
km-speech-corpus training).

## Why LoRA, not full fine-tuning
Instead of updating all ~242M of Whisper-small's parameters (expensive, needs
more GPU memory than a free-tier T4 comfortably offers), LoRA freezes the base
model completely and trains small "adapter" matrices injected into the attention
layers. Typically well under 1% of total parameters actually get trained, while
still meaningfully adapting the model's behavior — this is what makes fine-tuning
feasible at all on free Colab.

## Configuration used
- **Base checkpoint:** `seanghay/whisper-small-khmer-v2`
- **LoRA target modules:** `q_proj`, `v_proj` (standard choice for Whisper attention layers)
- **LoRA rank (r):** 16 | **alpha:** 32 | **dropout:** 0.05
- **Training data:** `data/processed/training_manifest.tsv` (68,059 new samples,
  deduplicated + shuffled, seed=42)
- **Precision:** fp16 (mixed precision, reduces T4 memory usage)
- **Checkpointing:** saved to Google Drive every 200 steps (see `ML-02`), to
  survive Colab disconnects without losing training progress

## Related tasks
`[ML-01] Set Up LoRA Fine-Tuning Script`, `[ML-02] Implement Drive Checkpointing`,
`[ML-03] Run Fine-Tuning` (see Trello board)

# Mount the Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/khmer-stt-bot'
print("Drive mounted. Project folder ready at:", DRIVE_PROJECT_DIR)

# **Load the Library**

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor, Seq2SeqTrainingArguments
from peft import LoraConfig, get_peft_model
import torch
from torch.utils.data import Dataset
import soundfile as sf
import os
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import Seq2SeqTrainer
from transformers.trainer_utils import get_last_checkpoint


# **Load the model and Configuration**

In [ ]:
!pip install -q -U torchao

In [ ]:
MODEL_ID = "seanghay/whisper-small-khmer-v2"

# Load base model + processor (same as DATA-02, reused here)
processor = WhisperProcessor.from_pretrained(MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

# LoRA configuration — this is the "sticky note" setup
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],  # standard choice for Whisper attention
    lora_dropout=0.05,
    bias="none",
)

# Wrap the base model with the LoRA adapter
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # shows you exactly how few params are actually trainable

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=f"{DRIVE_PROJECT_DIR}/models/checkpoints/lora_run1",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,   # effective batch size = 16
    learning_rate=1e-4,               # slightly higher than typical full fine-tune LR — normal for LoRA
    warmup_steps=50,
    num_train_epochs=3,
    fp16=True,                       # mixed precision, saves T4 memory
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=200,                  # tie into ML-02's checkpointing
    eval_steps=200,
    logging_steps=50,
    report_to="none",
)

## Step: Drive Checkpointing & Resume Logic

Build the training dataset from `training_manifest.tsv`, wire up the Trainer, and
implement resume-from-checkpoint logic so a Colab disconnect during the actual
fine-tuning run (`ML-03`) doesn't cost hours of progress — following the same
resume-safe pattern used throughout data preparation (DATA-05, DATA-08).

**Approach:** `output_dir` and `save_steps=200` (set in `ML-01`) already make the
`Trainer` save checkpoints to Google Drive periodically. What's added here: the
dataset loader, the data collator, and logic to detect and resume from the most
recent saved checkpoint automatically.

## Related task
`[ML-02] Implement Drive Checkpointing` (see Trello board)

# **Build a Dataset class from the manifest**

In [ ]:
manifest_path = f"{DRIVE_PROJECT_DIR}/data/processed/training_manifest.tsv"
DDD_RAW_DIR = f"{DRIVE_PROJECT_DIR}/data/raw/ddd_cambodia"
AUGMENTED_DIR = f"{DRIVE_PROJECT_DIR}/data/processed/ddd_augmented"

In [ ]:
class KhmerASRDataset(Dataset):
    def __init__(self, manifest_path, ddd_raw_dir, ddd_augmented_dir, processor):
        self.entries = []
        with open(manifest_path, encoding="utf-8") as f:
            for line in f:
                parts = line.rstrip("\n").split("\t")
                if len(parts) < 3:
                    continue
                filename, text, source = parts
                folder = ddd_raw_dir if source == "ddd_raw" else ddd_augmented_dir
                self.entries.append((os.path.join(folder, filename), text))
        self.processor = processor

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        audio_path, text = self.entries[idx]
        audio, sr = sf.read(audio_path)

        input_features = self.processor(
            audio, sampling_rate=sr, return_tensors="pt"
        ).input_features[0]

        labels = self.processor.tokenizer(
          text, return_tensors="pt", max_length=448, truncation=True
        ).input_ids[0]

        return {"input_features": input_features, "labels": labels}


In [ ]:
train_dataset = KhmerASRDataset(
    manifest_path=manifest_path,
    ddd_raw_dir=DDD_RAW_DIR,
    ddd_augmented_dir=AUGMENTED_DIR,
    processor=processor,
)
print(f"Training dataset ready: {len(train_dataset)} samples")

In [ ]:
# Check how many transcripts actually exceed the limit, before deciding
long_count = 0
for _, text, _ in [line.rstrip("\n").split("\t") for line in open(manifest_path, encoding="utf-8") if len(line.split("\t")) == 3]:
    token_count = len(processor.tokenizer(text).input_ids)
    if token_count > 448:
        long_count += 1
print(f"Transcripts exceeding 448 tokens: {long_count} out of {len(train_dataset)}")

## **Data collator (handles padding for variable-length audio/text)**

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Replace padding token id with -100 so it's ignored in the loss calculation
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

## **Build the Trainer, with resume logic**

In [ ]:
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
)

In [ ]:
# Check if a checkpoint already exists in output_dir (e.g. from a previous,
# interrupted run) — if so, resume from it instead of starting fresh
last_checkpoint = None
if os.path.isdir(training_args.output_dir):
    last_checkpoint = get_last_checkpoint(training_args.output_dir)
    if last_checkpoint is not None:
        print(f"Found existing checkpoint — will resume from: {last_checkpoint}")
    else:
        print("No checkpoint found — starting fresh")

## **Test if using GPU or not**

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("Model device:", next(model.parameters()).device)

## **Test with a short run (a few steps only — this is just a smoke test, not real training)**

In [ ]:
# Quick test: temporarily limit to a tiny number of steps just to confirm
# the whole pipeline runs and saves a checkpoint correctly
training_args.max_steps = 10
training_args.save_steps = 5

trainer.train(resume_from_checkpoint=last_checkpoint)

In [ ]:
# Bump step count so there's room to interrupt partway through
training_args.max_steps = 30
training_args.save_steps = 10

trainer.train(resume_from_checkpoint=last_checkpoint)

In [ ]:
import os
print(os.listdir(training_args.output_dir))

In [ ]:
last_checkpoint = get_last_checkpoint(training_args.output_dir)
print("Resuming from:", last_checkpoint)

trainer.train(resume_from_checkpoint=last_checkpoint)

## Step: Run Full LoRA Fine-Tuning

Execute the actual fine-tuning run on the full 68,059-sample training set
(56,716 raw DDD-Cambodia + 11,343 augmented). Estimated ~14 hours total compute
at ~4 sec/step — exceeds a single Colab session's ~12h cap, so this run spans
multiple sessions using the resume-from-checkpoint logic verified in `ML-02`.

**Plan:** run in Colab, monitor loss/stability, checkpoint every 200 steps.
If Colab's free-tier quota is exhausted, continue on Kaggle Notebooks
(30 GPU hrs/week) as a backup, loading the latest Drive checkpoint there.

## Related task
`[ML-03] Run Fine-Tuning` (see Trello board)

# **Reset training args to real values (undo the test-run overrides)**

In [ ]:
training_args.max_steps = -1          # let num_train_epochs control length, not a step cap
training_args.num_train_epochs = 3
training_args.save_steps = 200
training_args.eval_steps = 200
training_args.logging_steps = 50

# **Launch the Fine Tune Training run**

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(training_args.output_dir)
if last_checkpoint:
    print(f"Resuming from: {last_checkpoint}")
else:
    print("Starting fresh training run")

trainer.train(resume_from_checkpoint=last_checkpoint)

## Export Final LoRA Adapter

Explicitly save the final trained LoRA adapter weights (not the full base model) to a
clean, dedicated location, separate from the periodic training checkpoints — this
becomes the canonical model artifact referenced throughout evaluation and deployment.

In [ ]:
FINAL_ADAPTER_DIR = f"{DRIVE_PROJECT_DIR}/models/checkpoints/final_adapter"

trainer.save_model(FINAL_ADAPTER_DIR)
processor.save_pretrained(FINAL_ADAPTER_DIR)

print("Saved to:", FINAL_ADAPTER_DIR)

### Verify what actually got saved:

In [ ]:
import os
print(os.listdir(FINAL_ADAPTER_DIR))